In [ ]:
%%capture
import os
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from dj_notebook import activate

# local connection
env_file = os.environ["META_ENV"]

# remote connection
# open a tunnel first if using META_ENV_REMOTE
# ssh -N my.db.server -L 127.0.0.1:3307:127.0.0.1:3306
# env_file = os.environ["META_ENV_REMOTE"]

reports_folder = Path(os.environ["META_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["META_ANALYSIS_FOLDER"])
pharmacy_folder = Path(os.environ["META_PHARMACY_FOLDER"])
plus = activate(dotenv_file=env_file)


In [ ]:
from django_pandas.io import read_frame
from meta_subject.models import HivExitReview
from meta_lists.models import ArvRegimens
from clinicedc_constants import NULL_STRING


In [ ]:
df = read_frame(HivExitReview.objects.all(), verbose=False)
df_meds = read_frame(ArvRegimens.objects.all(), verbose=False)
df_meds = df_meds.rename(columns={"id": "current_arv_regimen"})

df = df.merge(
    df_meds[["current_arv_regimen", "name", "display_name"]], on="current_arv_regimen")
df = df.rename(columns={"display_name": "art_display_name", "name": "art_name"})

In [ ]:
df

In [ ]:
# df[["art_display_name", "art_name", "other_current_arv_regimen"]]
# df["art_name"].value_counts()
df[(~df["other_current_arv_regimen"].isna()) & (
        df["other_current_arv_regimen"] != NULL_STRING)][
    "other_current_arv_regimen"]

In [ ]:
df_meds

In [ ]:
from clinicedc_constants import OTHER
from meta_prn.models import OffStudyMedication

df = read_frame(OffStudyMedication.objects.all(), verbose=False)
df[df["reason"] == OTHER]["reason_other"].value_counts()

In [ ]:
reasons = [
    "Patient has reached month 48",
    "end of study",
    "Participant has reached month48 follow up",
    "The patient has completed three years and does not wish to reconsent",
    "Participant has reached month 48-month", "completed 36 months follow up",
    "completed 36 months of follow up",
    "Patient has reached month 36 and has declined to re-consent",
    "Reached 36 months refused to re consent",
    "reached month 48",
    "Patient reached month 36 asked time to think before she reconsent but eventually decided not to continue with the trial",
    "Completed 36 months of treatment",
    "Reached month 36 does not wish to continue with the trial",
    "reached month 36 and declined reconsenting",
    "Patient. has reach month 48",
]

In [ ]:
df["reason"].value_counts()


In [ ]:
from edc_offstudy.constants import COMPLETED_FOLLOWUP

df[df["reason"]==COMPLETED_FOLLOWUP].comment